# Gene ↔ ChemicalEntity Relation-Wise Merge

Merges Gene–Chemical triples from DRKG, PrimeKG, PharmKG, TARKG, Harmonizome, and hald;
resolves chemical tail names via PubChem (and DrugBank for DB-prefixed IDs) and gene head
names via NCBI; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [9]:
import os
import pandas as pd
import re

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'


OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_CHEMICALENTITY/ALL_GENE_CHEMICALENTITY.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical — PubChem and DrugBank

In [10]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} | DrugBank dict: {len(Drugbank_dict):,}")

PubChem IUPAC dict: 119,177,440 | DrugBank dict: 16,575


### 1b. Gene — NCBI and ENSEMBL

In [11]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


## 2. Load KG Sources

### 2a. DRKG

In [12]:
DRKG_Gene_Chemical = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Gene_Drug.csv')
DRKG_Gene_Chemical.columns = DRKG_Gene_Chemical.columns.str.lower()
DRKG_Gene_Chemical['tail_detail_name'] = DRKG_Gene_Chemical['tail'].astype(str).map(Pubchem_IUPAC_CID_Dict)
print(f"DRKG:        {len(DRKG_Gene_Chemical):,} rows")

DRKG_Gene_Chemical['kg_type'] = 'Generalised'
DRKG_Gene_Chemical['kg_source'] = 'DRKG'
DRKG_Gene_Chemical['species'] = 'Homo species'
DRKG_Gene_Chemical.head(2)

DRKG:        16,621 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_name,head_detail_name,head_ens,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,CDK7,Gene_Compound,3025986,Gene,DGIDB::INHIBITOR::Gene:Compound,Compound,DRKG,1022,cyclin dependent kinase 7,ENSG00000134058,"N-[5-[(5-tert-butyl-1,3-oxazol-2-yl)methylsulf...",NCBI_ID,Pubchem_ID,Gene::1022,Compound::DB05969,Generalised,Homo species
1,APOE,Gene_Compound,5865,Gene,DGIDB::OTHER::Gene:Compound,Compound,DRKG,348,apolipoprotein E,ENSG00000130203,"(8S,9S,10R,13S,14S,17R)-17-hydroxy-17-(2-hydro...",NCBI_ID,Pubchem_ID,Gene::348,Compound::DB00635,Generalised,Homo species


### 2b. PrimeKG

In [13]:
PrimeKG_Gene_Chemical = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Gene_ChemicalEntity.csv')
PrimeKG_Gene_Chemical.columns = PrimeKG_Gene_Chemical.columns.str.lower()
PrimeKG_Gene_Chemical['tail_detail_name'] = PrimeKG_Gene_Chemical['tail'].astype(str).map(Pubchem_IUPAC_CID_Dict)
print(f"PrimeKG:     {len(PrimeKG_Gene_Chemical):,} rows")
PrimeKG_Gene_Chemical['kg_type'] = 'Generalised'
PrimeKG_Gene_Chemical['kg_source'] = 'PrimeKG'
PrimeKG_Gene_Chemical['species'] = 'Homo species'
PrimeKG_Gene_Chemical.head(2)

PrimeKG:     25,312 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_alias_mapped,head_detail_name,head_ens,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species
0,F8,Gene_ChemicalEntity,23978,Gene,carrier,ChemicalEntity,PrimeKG,NCBI_Symbol,Pubchem,F8,coagulation factor VIII,ENSG00000185010,copper,23978,[Cu],[Cu],Generalised,Homo species
1,F5,Gene_ChemicalEntity,23978,Gene,carrier,ChemicalEntity,PrimeKG,NCBI_Symbol,Pubchem,F5,coagulation factor V,ENSG00000198734,copper,23978,[Cu],[Cu],Generalised,Homo species


In [21]:
PrimeKG_Gene_Chemical_1 = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Gene_Chemical_1.csv')
PrimeKG_Gene_Chemical_1.columns = PrimeKG_Gene_Chemical_1.columns.str.lower()
PrimeKG_Gene_Chemical_1['tail'] = (
    PrimeKG_Gene_Chemical_1['tail']
    .astype('Int64')   # handles NaN safely
    .astype(str)
)
PrimeKG_Gene_Chemical_1['tail_detail_name'] = PrimeKG_Gene_Chemical_1['tail'].astype(str).map(Pubchem_IUPAC_CID_Dict)
print(f"PrimeKG:     {len(PrimeKG_Gene_Chemical_1):,} rows")
PrimeKG_Gene_Chemical_1['kg_type'] = 'Generalised'
PrimeKG_Gene_Chemical_1['kg_source'] = 'PrimeKG'
PrimeKG_Gene_Chemical_1['species'] = 'Homo species'
PrimeKG_Gene_Chemical_1.head(2)

PrimeKG:     1,212 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_alias_mapped,head_detail_name,head_ens,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,kg_type,species
0,PPIAP10,Gene_Chemical,98490,Gene,interacts with,ChemicalEntity,PrimeKG,NCBI_Symbol,Pubchem,CRP,peptidylprolyl isomerase A pseudogene 10,ENSG00000224530,phenanthren-1-ol,1-hydroxyphenanthrene,C1=CC=C2C(=C1)C=CC3=C2C=CC=C3O,C1=CC=C2C(=C1)C=CC3=C2C=CC=C3O,Generalised,Homo species
1,ALCAM,Gene_Chemical,21387,Gene,interacts with,ChemicalEntity,PrimeKG,NCBI_Symbol,Pubchem,ALCAM,activated leukocyte cell adhesion molecule,ENSG00000170017,pyren-1-ol,1-hydroxypyrene,C1=CC2=C3C(=C1)C=CC4=C(C=CC(=C43)C=C2)O,C1=CC2=C3C(=C1)C=CC4=C(C=CC(=C43)C=C2)O,Generalised,Homo species


### 2c. PharmKG

In [14]:
PharmKG_Gene_Chemical = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Gene_Chemical.csv')
PharmKG_Gene_Chemical.columns = PharmKG_Gene_Chemical.columns.str.lower()
PharmKG_Gene_Chemical['tail_detail_name'] = PharmKG_Gene_Chemical['tail'].astype(str).map(Pubchem_IUPAC_CID_Dict)
PharmKG_Gene_Chemical['head_detail_name'] = PharmKG_Gene_Chemical['head'].astype(str).map(NCBI_ALL_Symb_Desc_dict)
print(f"PharmKG:     {len(PharmKG_Gene_Chemical):,} rows")
PharmKG_Gene_Chemical['kg_type'] = 'Generalised'
PharmKG_Gene_Chemical['kg_source'] = 'PrimeKG'
PharmKG_Gene_Chemical['species'] = 'Homo species'
PharmKG_Gene_Chemical.head(2)

PharmKG:     30,895 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_id_is,tail_id_is,tail_orignal,pubmed_id,sentence_tokenized,tail_detail_name,kg_type,species
0,CYP21A2,Gene_Chemical,753,gene,H,chemical,PrimeKG,cytochrome P450 family 21 subfamily A member 2,NCBI_ID,Pubchem,star,"26515592.0, 28800627.0, 16835396.0, 22396488.0...","'Additionally , in NCI-H295 cells , PNU-74654 ...","propane-1,2,3-triol",Generalised,Homo species
1,VDAC2,Gene_Chemical,6426943,gene,L,chemical,PrimeKG,voltage dependent anion channel 2,NCBI_ID,Pubchem,kd,"21241999.0, 21241999.0, 21241999.0, 21241999.0...",'VDAC2 KD significantly delayed mitochondrial ...,"(2S)-2-[[(2S)-2,6-diaminohexanoyl]amino]butane...",Generalised,Homo species


### 2d. TARKG

In [15]:
TARKG_Gene_Chemical = pd.read_csv(PROC_DIR + 'TARKG/Gene_Compound.csv')
TARKG_Gene_Chemical.columns = TARKG_Gene_Chemical.columns.str.lower()
print(f"TARKG:       {len(TARKG_Gene_Chemical):,} rows")

TARKG_Gene_Chemical['kg_type'] = 'Generalised'
TARKG_Gene_Chemical['kg_source'] = 'TARKG'
TARKG_Gene_Chemical['species'] = 'Homo species'
TARKG_Gene_Chemical.head(2)

TARKG:       173,931 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,TOP2A,Gene_ChemicalEntity,4583,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,CC1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(...,7-fluoro-2-methyl-6-(4-methylpiperazin-1-yl)-1...,NCBI_ID,Pubchem,OpenBioLink-3737050,OpenBioLink,1,Generalised,Homo species
1,TOP2A,Gene_ChemicalEntity,149096,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,(2S)-7-fluoro-2-methyl-6-(4-methylpiperazin-1-...,NCBI_ID,Pubchem,OpenBioLink-3737010,OpenBioLink,1,Generalised,Homo species


### 2e. Harmonizome

In [16]:
Harmonizome_Gene_Chemical = pd.read_csv(PROC_DIR + 'harmonizome/Gene_ChemicalEntity_Harmonizome.csv')
Harmonizome_Gene_Chemical.columns = Harmonizome_Gene_Chemical.columns.str.lower()
print(f"Harmonizome: {len(Harmonizome_Gene_Chemical):,} rows")

Harmonizome_Gene_Chemical['kg_type'] = 'Generalised'
Harmonizome_Gene_Chemical['kg_source'] = 'Harmonizome'
Harmonizome_Gene_Chemical['species'] = 'Homo species'
Harmonizome_Gene_Chemical.head(2)

Harmonizome: 27,588 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,TBXA2R,Gene_ChemicalEntity,6438346,Gene,ChemicalEntity,CTD,Harmonizome,thromboxane A2 receptor,7-(3-(3-hydroxy-4-(4'-iodophenoxy)-1-butenyl)-...,NCBI_ID,Pubchem,Generalised,Homo species
1,TBXA2R,Gene_ChemicalEntity,5311448,Gene,ChemicalEntity,CTD,Harmonizome,thromboxane A2 receptor,SQ 29548,NCBI_ID,Pubchem,Generalised,Homo species


### 2f. hald

In [17]:
hald = pd.read_csv(PROC_DIR + 'hald/Gene_Chemical_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald:      {len(hald):,} rows")
hald['kg_type'] = 'Aging'
hald['kg_source'] = 'HALD_KG'
hald['species'] = 'Homo species'
hald.head(2)

hald:      820 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,NOX1,Gene_ChemicalEntity,DB00065,Gene,activated,ChemicalEntity,HALD_KG,NADPH oxidase 1,Infliximab,Infliximab,NCBI_ID,Drugbank,Aging,Homo species
1,NOX4,Gene_ChemicalEntity,DB00065,Gene,activated,ChemicalEntity,HALD_KG,NADPH oxidase 4,Infliximab,Infliximab,NCBI_ID,Drugbank,Aging,Homo species


### 2g. Biograkn

In [18]:
Biograkn_Gene_Chemical = pd.read_csv(PROC_DIR + 'BioGrakn/Gene_Chemical_1_Biogrkn.csv')
Biograkn_Gene_Chemical.columns = Biograkn_Gene_Chemical.columns.str.lower()
print(f"Biograkn: {len(Biograkn_Gene_Chemical):,} rows")

Biograkn_Gene_Chemical['kg_type'] = 'Generalised'

Biograkn_Gene_Chemical['species'] = 'Homo species'
Biograkn_Gene_Chemical.head(2)

Biograkn: 27,694 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,CDK7,Gene_ChemicalEntity,3025986,Gene,ChemicalEntity,BioGrakn,cyclin dependent kinase 7,"N-[5-[(5-tert-butyl-1,3-oxazol-2-yl)methylsulf...",NCBI_ID,Pubchem,Generalised,Homo species
1,ADORA2A,Gene_ChemicalEntity,10247549,Gene,ChemicalEntity,BioGrakn,adenosine A2a receptor,"3,5,7-triethoxy-2-phenylchromen-4-one",NCBI_ID,Pubchem,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [23]:
SOURCE_DFS = [
    ('PrimeKG_Gene_Chemical',     PrimeKG_Gene_Chemical),
    ('PrimeKG_Gene_Chemical_1',   PrimeKG_Gene_Chemical_1),
    ('PharmKG_Gene_Chemical',     PharmKG_Gene_Chemical),
    ('TARKG_Gene_Chemical',       TARKG_Gene_Chemical),
    ('Harmonizome_Gene_Chemical', Harmonizome_Gene_Chemical),
    ('hald',                    hald),
    ('Biograkn_Gene_Chemical',  Biograkn_Gene_Chemical),
    
]

for name, df in SOURCE_DFS:
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[PrimeKG_Gene_Chemical] ✓ no duplicates
[PrimeKG_Gene_Chemical_1] ✓ no duplicates
[PharmKG_Gene_Chemical] ✓ no duplicates
[TARKG_Gene_Chemical] ✓ no duplicates
[Harmonizome_Gene_Chemical] ✓ no duplicates
[hald] ✓ no duplicates
[Biograkn_Gene_Chemical] ✓ no duplicates


In [24]:
SOURCE_DFS = [(name, df.loc[:, ~df.columns.duplicated(keep='first')]) for name, df in SOURCE_DFS]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[PrimeKG_Gene_Chemical] ✓ clean
[PrimeKG_Gene_Chemical_1] ✓ clean
[PharmKG_Gene_Chemical] ✓ clean
[TARKG_Gene_Chemical] ✓ clean
[Harmonizome_Gene_Chemical] ✓ clean
[hald] ✓ clean
[Biograkn_Gene_Chemical] ✓ clean


## 4. Align Schemas and Concatenate

Note: DRKG is excluded from the final merge, matching the original notebook's source list.

In [25]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 287,452 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,F8,Gene_ChemicalEntity,23978,Gene,carrier,ChemicalEntity,PrimeKG,NCBI_Symbol,Pubchem,coagulation factor VIII,copper,Generalised,Homo species
1,F5,Gene_ChemicalEntity,23978,Gene,carrier,ChemicalEntity,PrimeKG,NCBI_Symbol,Pubchem,coagulation factor V,copper,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [26]:
final_df['relation']   = 'Gene_ChemicalEntity'
final_df['head_type']  = 'Gene'
final_df['tail_type']  = 'ChemicalEntity'
final_df['head_id_is'] = 'NCBI_ID'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))
print("Unique kg_source:",  set(final_df['kg_type']))


Unique relation: {'Gene_ChemicalEntity'}
Unique head_id_is: {'NCBI_ID'}
Unique tail_id_is: {'Pubchem', nan, 'Drugbank'}
Unique kg_source: {'HALD_KG', 'Harmonizome', 'TARKG', 'PrimeKG', 'BioGrakn'}
Unique kg_source: {'Generalised', 'Aging'}


## 6. Deduplicate

In [27]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
        'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 255,771 rows


## 7. Repair Gene Head Names and Resolve Chemical Tail Names

1. Repair missing gene head names via synonym + NCBI description.
2. Normalise DrugBank tail IDs to zero-padded 5-digit; DrugBank name fallback.
3. Drop rows with no `tail_detail_name`.

In [33]:
#

In [28]:
mask = final_df_dedup['head_detail_name'].isna()
print(f"Rows needing head_detail_name repair: {mask.sum():,}")
final_df_dedup.loc[mask, 'head'] = final_df_dedup.loc[mask, 'head'].str.replace('-', '', regex=False)
final_df_dedup.loc[mask, 'head'] = (
    final_df_dedup.loc[mask, 'head']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df_dedup.loc[mask, 'head'])
)
mask2 = final_df_dedup['head_detail_name'].isna()
final_df_dedup.loc[mask2, 'head_detail_name'] = final_df_dedup.loc[mask2, 'head'].map(NCBI_ALL_Symb_Desc_dict)
print(f"Still missing head_detail_name: {final_df_dedup['head_detail_name'].isna().sum():,}")

# Normalise DrugBank tail IDs and fill names
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['tail'] = final_df_dedup['tail'].apply(standardize_drugbank_id).astype(str)
mask = final_df_dedup['tail_detail_name'].isna() & final_df_dedup['tail'].str.startswith('DB')
final_df_dedup.loc[mask, 'tail_detail_name'] = final_df_dedup.loc[mask, 'tail'].map(Drugbank_dict)

# Drop rows with no tail name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing tail_detail_name → {len(final_df_dedup):,} remaining")

Rows needing head_detail_name repair: 850
Still missing head_detail_name: 1
Dropped 919 rows with missing tail_detail_name → 254,852 remaining


## 8. Add Schema Columns and Enforce Column Order

In [29]:
final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (254852, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,PGD,Gene_ChemicalEntity,10267025,Gene,E+,ChemicalEntity,PrimeKG,NCBI_ID,Pubchem,phosphogluconate dehydrogenase,"(5R,6R,7S,8S)-5-(hydroxymethyl)-2-(2-phenyleth...",Generalised,Homo species
1,PGD,Gene_ChemicalEntity,68384915,Gene,E+,ChemicalEntity,PrimeKG,NCBI_ID,Pubchem,phosphogluconate dehydrogenase,"lithium;phenyl-(2,4,6-trimethylbenzoyl)phosphi...",Generalised,Homo species
2,A1BG,Gene_ChemicalEntity,11192,Gene,target,ChemicalEntity,PrimeKG,NCBI_ID,Pubchem,alpha-1-B glycoprotein,zinc;diacetate,Generalised,Homo species
3,A1BG,Gene_ChemicalEntity,23978,Gene,target,ChemicalEntity,PrimeKG,NCBI_ID,Pubchem,alpha-1-B glycoprotein,copper,Generalised,Homo species
4,A1BG,Gene_ChemicalEntity,23994,Gene,target,ChemicalEntity,PrimeKG,NCBI_ID,Pubchem,alpha-1-B glycoprotein,zinc,Generalised,Homo species


In [30]:
final_df_dedup[final_df_dedup['tail_id_is'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
3294,ACAN,Gene_ChemicalEntity,Glycosaminoglycans,Gene,reduced,ChemicalEntity,HALD_KG,NCBI_ID,None,aggrecan,Glycosaminoglycans,Aging,Homo species
3359,ACAT2,Gene_ChemicalEntity,Cholesterol Esters,Gene,suggest,ChemicalEntity,HALD_KG,NCBI_ID,None,acetyl-CoA acetyltransferase 2,Cholesterol Esters,Aging,Homo species
4765,ACHE,Gene_ChemicalEntity,Polysaccharides,Gene,differ,ChemicalEntity,HALD_KG,NCBI_ID,None,acetylcholinesterase (Yt blood group),Polysaccharides,Aging,Homo species
6873,ADARB2,Gene_ChemicalEntity,Triglycerides,Gene,associated,ChemicalEntity,HALD_KG,NCBI_ID,None,adenosine deaminase RNA specific B2 (inactive),Triglycerides,Aging,Homo species
8353,ADIPOQ,Gene_ChemicalEntity,Ceramides,Gene,circulate,ChemicalEntity,HALD_KG,NCBI_ID,None,"adiponectin, C1Q and collagen domain containing",Ceramides,Aging,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
248946,UCP2,Gene_ChemicalEntity,Fatty Acids,Gene,maintain,ChemicalEntity,HALD_KG,NCBI_ID,None,uncoupling protein 2,Fatty Acids,Aging,Homo species
248962,UCP3,Gene_ChemicalEntity,Fatty Acids,Gene,catalyse,ChemicalEntity,HALD_KG,NCBI_ID,None,uncoupling protein 3,Fatty Acids,Aging,Homo species
252805,VEGFA,Gene_ChemicalEntity,Lipopolysaccharides,Gene,measured,ChemicalEntity,HALD_KG,NCBI_ID,None,vascular endothelial growth factor A,Lipopolysaccharides,Aging,Homo species
252999,VIP,Gene_ChemicalEntity,Lipopolysaccharides,Gene,inhibit,ChemicalEntity,HALD_KG,NCBI_ID,None,vasoactive intestinal peptide,Lipopolysaccharides,Aging,Homo species


## 9. QC — NaN Check

In [31]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,28974,0,28974
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,180,0,180
head_detail_name,0,0,0


## 10. Save Output

In [32]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 254,852 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_CHEMICALENTITY/ALL_GENE_CHEMICALENTITY.parquet


In [34]:
#